# ClinVar / TCGA / 1000G tutorial

This notebook uses **ClinVar / TCGA / 1000 Genomes** variant-window FASTA files as an example and walks through the
`variant_aware.py` **catalog variants** workflow (also referred to as `--genomic_variants`).

You will find three kinds of information here:

1. **What the input looks like**: which tokens must appear in the FASTA header, how long the window sequence should be, and how to write `POS:REF>ALT`.
2. **What the output looks like**: for each record, one line is appended to the output file, including `model_id`, `mode`, and `Prediction_score`.
3. **How the code flows**: the later code cells decompose the script into modules, and add deeper documentation (docstrings / comments / expected I/O) at key steps.

> Note: End-to-end scoring loads both the Transformer and the BRIDGE checkpoint, so you need to run this notebook in a properly
> configured **BRIDGE** environment (dependencies installed, and the model paths available).  
> This notebook also includes a *lightweight demo* to first validate **header parsing and variant localization**.


## CLI arguments

### Common arguments
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--variation_mode` | `before|after` | yes | Score reference window ('before') or ALT-substituted window ('after'). |
| `--fasta_sequence_path` | `PATH` | yes | Input FASTA containing window sequences. |
| `--variant_out_file` | `PATH` | yes | Output file (appended). |
| `--Transformer_path` | `PATH` | yes | Transformer path used by build_Transformer_embeddings. |
| `--model_save_path` | `PATH` | yes | Directory with BRIDGE checkpoints (*.pth). |
| `--device` | `cuda|cuda:N|cpu` | no | Torch device (default: cuda if available else cpu). |

### Pipeline selection flags
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--gwas` | `-` | no | Force GWAS pipeline (default if no pipeline flag provided). |
| `--ribosnitch/--ribosnitches` | `-` | no | Enable ribosnitch pipeline. |
| `--catalog_variants/--genomic_variants` | `-` | no | Enable catalog-variants pipeline (ClinVar/TCGA/1000G). |

### Catalog-variants-only arguments
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--model_id_strategy` | `from_header|from_fasta_stem` | no | Catalog mode: choose checkpoint naming strategy. |
| `--k` | `int` | no | Catalog mode: forwarded to build_Transformer_embeddings. |
| `--pos_weight` | `float` | no | Catalog mode: pos_weight for BCEWithLogitsLoss. |
| `--strict_ref_match` | `-` | no | Catalog mode: skip if REF/ALT cannot be matched in the window. |
| `--disable_off_by_one` | `-` | no | Catalog mode: disable +/-1 fallback when locating SNV index. |

### Quickstart (the two most common commands)

Below is an example using a 1000G FASTA file (typically you run the same FASTA twice: `before` and `after`):

```bash
python variant_aware.py \
  --catalog_variants \
  --variation_mode before \
  --fasta_sequence_path /path/to/1000genomes_diff.all.fa \
  --variant_out_file ./results/variant/mut_before_after_score/1000genomes.before.txt \
  --Transformer_path /path/to/RBPformer \
  --model_save_path ./results/model \
  --device cpu
```

```bash
python variant_aware.py \
  --catalog_variants \
  --variation_mode after \
  --fasta_sequence_path /path/to/1000genomes_diff.all.fa \
  --variant_out_file ./results/variant/mut_before_after_score/1000genomes.after.txt \
  --Transformer_path /path/to/RBPformer \
  --model_save_path ./results/model \
  --device cuda:0
```

**What you should see in the terminal logs** typically includes:

- The number of FASTA records and the window length (if you enable the sanity-check cell below).
- Per-record header parsing (errors/warnings will be printed if a header cannot be parsed).
- A final message like "Results appended to ...".


## Input FASTA requirements (Catalog variants)

- Header must include `chr:start-end(strand)` and `POS:REF>ALT`.
- Default model id parsing expects `... <PROTEIN> in <CELL>`; otherwise set `--model_id_strategy from_fasta_stem`.

### Example input (recommended: copy and edit)

A typical header should contain at least two required tokens:

- **Region token**: `chr:start-end(strand)`, e.g. `chr1:100-200(+)`
- **Variant token**: `POS:REF>ALT`, e.g. `150:A>G`

Optionally (but commonly), include protein and cell line information so the default checkpoint id can be inferred as
`model_id = <PROTEIN>_<CELL>`:

```fasta
>chr1:100-200(+) 150:A>G SRSF1 in K562
ACGTT... (a fixed-length window sequence, e.g. 101 nt)
```

#### What should the sequence length be?

Many BRIDGE checkpoints (and the feature tensors created by this script) are designed around a **101 nt window**
(you will see `101` in the code).  
If your trained/downloaded checkpoint uses a different window length, make sure your input sequence length matches the checkpoint;
otherwise the embedding/feature dimensions may not align.

#### What happens when `strand` is `-`?

When `strand == '-'`, the script complements `REF/ALT` before locating/replacing the variant so that the *sequence orientation*
matches what the model expects (consistent with how the model was trained).


## Output format (Catalog variants)

```
<header_without_>\tmodel_id=<...>\tmode=<before|after>\tPrediction_score:<float>
```

### Example output (single line)

```text
chr1:100-200(+) 150:A>G SRSF1 in K562	model_id=SRSF1_K562	mode=after	Prediction_score:0.123456
```

- `mode=before`: score the *as-is* window sequence (if the input already contains ALT at the variant position, the script will log it and still score the sequence as provided).
- `mode=after`: locate the variant position in the window and replace it with ALT before scoring.
- `Prediction_score`: a floating-point model score (in the current implementation, this is the logits/score returned by `validate_without_sigmoid`).


## Lightweight demo: make the input → parsing → output shape visible

This section does three things:

1. Parse the header into `chrom/start/end/strand/var_pos/ref/alt/protein/cell_line`
2. Locate the variant base in the window sequence as a **0-based index**
3. Print what a *final output line* looks like (the score is a placeholder)

> Goal: make the input/output and key intermediate variables explicit, so the end-to-end model scoring is not a "black box".


In [1]:
import re
from dataclasses import dataclass
from typing import Optional, List, Tuple

# --- Minimal, self-contained demo of the header parsing logic used by the catalog-variants pipeline ---

_REGION_RE = re.compile(r"^(chr[^:]+):(\d+)-(\d+)\(([+-])\)")
_VAR_RE = re.compile(r"^(\d+):([ACGTN])>([ACGTN])$")

@dataclass
class ParsedHeader:
    header_raw: str
    chrom: str
    start: int
    end: int
    strand: str
    var_pos: int
    ref: str
    alt: str
    protein: Optional[str] = None
    cell_line: Optional[str] = None

def parse_protein_cell_line(fields: List[str]) -> Tuple[Optional[str], Optional[str]]:
    # Heuristic: "... <PROTEIN> in <CELL>"
    if "in" in fields:
        idx_in = len(fields) - 1 - list(reversed(fields)).index("in")
        protein = fields[idx_in - 1] if idx_in - 1 >= 0 else None
        cell = fields[idx_in + 1] if idx_in + 1 < len(fields) else None
        return protein, cell
    return None, None

def parse_header_catalog(header_raw: str) -> ParsedHeader:
    fields = header_raw.split()
    region_tok = next((t for t in fields if _REGION_RE.match(t)), None)
    if region_tok is None:
        raise ValueError("missing region token like chr:start-end(strand)")
    var_tok = next((t for t in fields if _VAR_RE.match(t)), None)
    if var_tok is None:
        raise ValueError("missing SNV token like POS:REF>ALT")

    m_r = _REGION_RE.match(region_tok)
    chrom, start, end, strand = m_r.group(1), int(m_r.group(2)), int(m_r.group(3)), m_r.group(4)

    m_v = _VAR_RE.match(var_tok)
    var_pos, ref, alt = int(m_v.group(1)), m_v.group(2), m_v.group(3)

    protein, cell = parse_protein_cell_line(fields)
    return ParsedHeader(header_raw, chrom, start, end, strand, var_pos, ref, alt, protein, cell)

def find_variant_index(seq: str, seq_start: int, var_pos: int, ref: str, alt: str, try_off_by_one: bool=True) -> Tuple[Optional[int], str]:
    candidates = [var_pos - seq_start]
    if try_off_by_one:
        candidates.append(var_pos - seq_start - 1)
    for idx0 in candidates:
        if 0 <= idx0 < len(seq):
            if seq[idx0] == ref:
                return idx0, "ref"
            if seq[idx0] == alt:
                return idx0, "alt"
    return None, "none"

def substitute_base(seq: str, idx0: int, base: str) -> str:
    return seq[:idx0] + base + seq[idx0+1:]

# Example header and 101-nt window (commonly used by BRIDGE checkpoints)
example_header = "chr1:100-200(+) 150:A>G SRSF1 in K562"
ph = parse_header_catalog(example_header)

# Make a toy 101-nt sequence where the REF base is at the expected position
window_len = 101
idx_expected = ph.var_pos - ph.start
seq = ("C" * window_len)
seq = substitute_base(seq, idx_expected, ph.ref)  # put REF base at the variant site

idx0, state = find_variant_index(seq, ph.start, ph.var_pos, ph.ref, ph.alt)
mut = substitute_base(seq, idx0, ph.alt) if idx0 is not None else seq

print("ParsedHeader:", ph)
print("Window length:", len(seq))
print("Variant index (0-based in window):", idx0, "| match_state:", state)
print("Base at site (before/after):", seq[idx0], "->", mut[idx0])
print("Example output line:")
print(f"{ph.header_raw}\tmodel_id={ph.protein}_{ph.cell_line}\tmode=after\tPrediction_score:0.123456")

ParsedHeader: ParsedHeader(header_raw='chr1:100-200(+) 150:A>G SRSF1 in K562', chrom='chr1', start=100, end=200, strand='+', var_pos=150, ref='A', alt='G', protein='SRSF1', cell_line='K562')
Window length: 101
Variant index (0-based in window): 50 | match_state: ref
Base at site (before/after): A -> G
Example output line:
chr1:100-200(+) 150:A>G SRSF1 in K562	model_id=SRSF1_K562	mode=after	Prediction_score:0.123456


## Script code (identical logic to the repository script)

From here on, the cells contain the **script code itself** (the relevant parts of `variant_aware.py`), with two upgrades:

- **Explanations before each code block**: what the block reads, what it produces, and how failures are handled.
- **Deeper documentation**: added docstrings (Args/Returns/Notes) for key functions, plus inline comments at common pitfalls.

> Preconditions:
> - It is recommended to start this notebook from the BRIDGE repository root (or a subdirectory) so imports like `utils/` resolve correctly.
> - End-to-end scoring requires: `--Transformer_path` pointing to the Transformer (e.g. `RBPformer`) directory, and `--model_save_path` pointing to the `.pth` checkpoint directory.


In [2]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import argparse
import logging
import os
import re
import sys
from pathlib import Path
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Callable

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel

repo_root = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(repo_root))

from utils.BRIDGE import BRIDGE
from utils.gen_transformer_embedding import build_Transformer_embeddings
from utils.train_loop import validate_without_sigmoid
from utils.utils import RBPInferDataset
from utils.FeatureEncoding import dealwithdata2
from utils.variant import read_fasta, open_output, parse_variant_block, apply_complement, substitute_base, ModelHub, parse_variant_block_flexible

# -----------------------------------------------------------------------------
# What do these imported modules do? (high-level)
#
# - utils.variant:
#     * read_fasta(path) -> (headers, sequences)
#     * open_output(path) -> Path with parent dirs ensured
#     * substitute_base(seq, idx0, base) -> mutated sequence string
#     * ModelHub(transformer_path, device): loads Transformer + BRIDGE checkpoints
#
# - utils.gen_transformer_embedding.build_Transformer_embeddings:
#     Encodes sequences into model-ready embeddings (k-mer / Transformer features).
#
# - utils.BRIDGE.BRIDGE:
#     Core BRIDGE model class (loaded from checkpoint).
#
# - utils.train_loop.validate_without_sigmoid:
#     Convenience wrapper to run inference and return a scalar score (logit-like).
#
# - utils.FeatureEncoding.dealwithdata2:
#     Hand-crafted feature encoder (e.g., biochemical channels) used by BRIDGE.
# -----------------------------------------------------------------------------

/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Catalog variants: header parsing, variant localization, and window rewriting

This code corresponds to the `--catalog_variants` / `--genomic_variants` branch and defines three key helpers:

1. `parse_header_catalog(header_raw) -> ParsedHeader`  
   - **Input**: FASTA header line (without the leading `>`)  
   - **Output**: parsed fields `chrom/start/end/strand/var_pos/ref/alt/protein/cell_line`

2. `find_variant_index(seq, seq_start, var_pos, ref, alt) -> (idx0, state)`  
   - **Input**: window sequence plus the coordinates/alleles from the header  
   - **Output**: the 0-based variant index `idx0` (or `None` if not found) and a match state (`"ref"`, `"alt"`, or `"none"`)

3. (Used later during scoring) choose between using the original sequence (`before`) or replacing with ALT (`after`) based on `--variation_mode`.

> Run the "lightweight demo" first to understand these intermediate variables, then proceed to the end-to-end model scoring.


In [3]:

# ClinVar / TCGA / 1000 Genomes style FASTA batches ("catalog variants")
# ---------------------------------------------------------------------------
# These datasets typically store variant windows directly in FASTA, where the header
# contains (chrom:start-end(strand)) and a SNV token like POS:REF>ALT.
#
# This branch is activated by passing `--catalog_variants` (alias: --genomic_variants).
# The default model naming strategy is `<PROTEIN>_<CELL>` parsed from the header.
# ---------------------------------------------------------------------------

_REGION_RE = re.compile(r"^(chr[^:]+):(\d+)-(\d+)\(([+-])\)")
_VAR_RE = re.compile(r"^(\d+):([ACGTN])>([ACGTN])$")


@dataclass
class ParsedHeader:
    "Parsed representation of a variant-window FASTA header."
    header_raw: str
    chrom: str
    start: int
    end: int
    strand: str
    var_pos: int
    ref: str
    alt: str
    protein: Optional[str]
    cell_line: Optional[str]


def parse_protein_cell_line(fields: List[str]) -> Tuple[Optional[str], Optional[str]]:
    """
    Heuristically parse the `(protein, cell_line)` pair from a tokenized FASTA header.

    This function supports the default `model_id` construction used by the catalog-variants
    pipeline: `model_id = <PROTEIN>_<CELL>`.

    Expected header patterns (space-separated tokens):
      - "... <PROTEIN> in <CELL>"      (preferred)
      - otherwise returns `(None, None)` and the caller can fall back to
        `--model_id_strategy from_fasta_stem`.

    Args:
        fields: `header_raw.split()`.

    Returns:
        A tuple `(protein, cell_line)`. Either element can be None if not found.

    Notes:
        This is a **heuristic** parser by design; headers from different catalogs may
        have different token orders. Keep your headers consistent within one FASTA.
    """
    protein: Optional[str] = None
    cell: Optional[str] = None

    # Prefer "... <PROTEIN> in <CELL>"
    if "in" in fields:
        # use the last "in" to be robust if "in" appears elsewhere
        idx_in = len(fields) - 1 - list(reversed(fields)).index("in")
        if idx_in + 1 < len(fields):
            cell = fields[idx_in + 1]
        if idx_in - 1 >= 0:
            protein = fields[idx_in - 1]

    # Fallbacks (match the old "[-3],[-1]" convention)
    if cell is None and len(fields) >= 1:
        cell = fields[-1]
    if protein is None:
        if len(fields) >= 3 and fields[-2] == "in":
            protein = fields[-3]
        elif len(fields) >= 3:
            protein = fields[-3]
        elif len(fields) >= 2:
            protein = fields[-2]

    return protein, cell


def parse_header_catalog(header_raw: str) -> ParsedHeader:
    """
    Parse a ClinVar/TCGA/1000G-style (catalog-variants) FASTA header.

    The parser looks for two required tokens anywhere in the header:

    1) region token:  `chr:start-end(strand)`  e.g. `chr1:100-200(+)`
    2) SNV token:     `POS:REF>ALT`            e.g. `150:A>G`

    It also tries to infer `protein` and `cell_line` using `parse_protein_cell_line()`.

    Args:
        header_raw: Full header string WITHOUT the leading '>'.

    Returns:
        A `ParsedHeader` dataclass containing:
          - genomic interval (chrom/start/end/strand)
          - variant triple (var_pos/ref/alt)
          - optional metadata (protein/cell_line)

    Raises:
        ValueError: If required tokens cannot be found or do not match expected patterns.
    """
    fields = header_raw.split()

    region_tok: Optional[str] = None
    var_tok: Optional[str] = None

    for tok in fields:
        if tok.startswith("chr") and ":" in tok and "(" in tok and ")" in tok:
            region_tok = tok
            break
    if region_tok is None:
        for tok in fields:
            if _REGION_RE.match(tok):
                region_tok = tok
                break
    if region_tok is None:
        raise ValueError(f"Cannot find region token like chr:start-end(strand) in header: {header_raw}")

    for tok in fields:
        if _VAR_RE.match(tok):
            var_tok = tok
            break
    if var_tok is None:
        raise ValueError(f"Cannot find SNV token like POS:REF>ALT in header: {header_raw}")

    m_r = _REGION_RE.match(region_tok)
    if m_r is None:
        raise ValueError(f"Region token doesn't match expected pattern: {region_tok}")
    chrom, start, end, strand = m_r.group(1), int(m_r.group(2)), int(m_r.group(3)), m_r.group(4)

    m_v = _VAR_RE.match(var_tok)
    assert m_v is not None
    var_pos, ref, alt = int(m_v.group(1)), m_v.group(2), m_v.group(3)

    protein, cell = parse_protein_cell_line(fields)

    return ParsedHeader(
        header_raw=header_raw,
        chrom=chrom,
        start=start,
        end=end,
        strand=strand,
        var_pos=var_pos,
        ref=ref,
        alt=alt,
        protein=protein,
        cell_line=cell,
    )


def find_variant_index(
    seq: str,
    seq_start: int,
    var_pos: int,
    ref: str,
    alt: str,
    try_off_by_one: bool = True,
) -> Tuple[Optional[int], str]:
    """
    Locate the 0-based index of the SNV position inside the window sequence.

    The intended mapping is:
        idx0 = var_pos - seq_start

    Some FASTA catalogs may use 1-based coordinates or have off-by-one inconsistencies,
    so the function can optionally try `idx0 - 1` as a fallback.

    Args:
        seq: Window sequence (string) as read from FASTA.
        seq_start: Window start coordinate parsed from header.
        var_pos: Variant genomic position parsed from header.
        ref: Reference allele to match (already complemented if strand == '-').
        alt: Alternate allele (already complemented if strand == '-').
        try_off_by_one: Whether to also test (var_pos - seq_start - 1).

    Returns:
        (idx0, state)
          - idx0: 0-based index inside `seq`, or None if not found.
          - state: "ref" if seq[idx0]==ref, "alt" if seq[idx0]==alt, otherwise "none".

    Notes:
        The caller can decide how strict to be using `--strict_ref_match`.
    """
    candidates = [var_pos - seq_start]
    if try_off_by_one:
        candidates.append(var_pos - seq_start - 1)

    for idx0 in candidates:
        if idx0 < 0 or idx0 >= len(seq):
            continue
        base = seq[idx0]
        if base == ref:
            return idx0, "ref"
        if base == alt:
            return idx0, "alt"

    return None, "none"


def process_sequences_catalog_variants(
    headers: List[str],
    seqs: List[str],
    args: argparse.Namespace,
    hub: ModelHub,
) -> None:
    "Variant-aware scoring for ClinVar/TCGA/1000G-style FASTA batches (SNVs)."
    out_fp = open_output(args.variant_out_file)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(args.pos_weight), device=hub.device))

    with out_fp.open("a") as fout:
        for header_raw, seq in zip(headers, seqs):
            try:
                ph = parse_header_catalog(header_raw)
            except Exception as e:
                logging.error("[catalog_variants] Header parse failed: %s | %s", header_raw, e)
                continue

            ref, alt = ph.ref, ph.alt
            if ph.strand == "-":
                ref, alt = apply_complement(ref), apply_complement(alt)

            idx0, state = find_variant_index(
                seq=seq,
                seq_start=ph.start,
                var_pos=ph.var_pos,
                ref=ref,
                alt=alt,
                try_off_by_one=(not bool(args.disable_off_by_one)),
            )
            if state == "none":
                logging.warning("[catalog_variants] Cannot match REF/ALT at site: %s", ph.header_raw)
                if bool(args.strict_ref_match):
                    continue

            # choose checkpoint name
            if args.model_id_strategy == "from_fasta_stem":
                model_id = Path(args.fasta_sequence_path).stem
            else:
                if not ph.protein or not ph.cell_line:
                    logging.warning("[catalog_variants] Cannot parse protein/cell line for model id: %s", ph.header_raw)
                    continue
                model_id = f"{ph.protein}_{ph.cell_line}"

            model = hub.load_bridge(Path(args.model_save_path), model_id)
            if model is None:
                continue

            # build modified_seq depending on variation_mode and what we see in input
            if args.variation_mode == "before":
                if state == "alt":
                    logging.info("[catalog_variants] Input already ALT at site; scoring as-is in BEFORE: %s", ph.header_raw)
                modified_seq = seq
            else:  # after
                if idx0 is None:
                    modified_seq = seq
                elif state == "ref":
                    modified_seq = substitute_base(seq, idx0, alt)
                else:
                    modified_seq = seq

            emb, _ = build_Transformer_embeddings(
                sequences=[modified_seq],
                transformer_path=str(args.Transformer_path),
                device=hub.device,
                k=int(args.k),
                transpose_to_ch_first=True,
            )
            N = int(emb.shape[0])

            attn = np.zeros((N, 101, 103))
            struct = np.zeros((N, 1, 101))
            motif = np.zeros((N, 1, 101))
            biochem = dealwithdata2(modified_seq).transpose([0, 2, 1])

            dataset = RBPInferDataset(
                embedding=emb,
                attn=attn,
                struct=struct,
                motif=motif,
                biochem=biochem,
            )
            loader = DataLoader(dataset, batch_size=1, shuffle=False)

            score = validate_without_sigmoid(model, hub.device, loader, criterion).item()

            # Output keeps genomic_variants.py style (adds model_id/mode)
            fout.write(
                f"{ph.header_raw}\tmodel_id={model_id}"
                f"\tmode={args.variation_mode}\tPrediction_score:{score:.6f}\n"
            )

### CLI and entrypoint: argument parsing, pipeline selection, and `main()`

This block contains:

- `build_argparser()`: defines all CLI arguments in one place (including catalog variants / ribosnitch / GWAS modes).
- `main()`: parse args → read FASTA → initialize `ModelHub` → select and run the appropriate pipeline based on flags.

> For this tutorial, focus on `--catalog_variants` (or `--genomic_variants`) and the related parameters such as `--model_id_strategy` and `--strict_ref_match`.


In [4]:
def build_argparser() -> argparse.ArgumentParser:
    """
    Build the CLI argument parser for `variant_aware.py`.

    This entry point supports multiple pipelines. In this notebook we focus on:
    - catalog variants: `--catalog_variants` (alias: `--genomic_variants`)

    Returns:
        An `argparse.ArgumentParser` with all supported flags and defaults.

    Tip:
        Run `print(build_argparser().format_help())` to see a full, rendered help message.
    """
    parser = argparse.ArgumentParser(
        description=(
            "Variant-aware scoring with BRIDGE. Supports three pipelines:\n"
            "  (1) GWAS windows (legacy variant_aware.py behavior; default)\n"
            "  (2) Ribosnitch scoring (BRIDGE; per-record checkpoint selection)\n"
            "  (3) Catalog variants (ClinVar/TCGA/1000G-style FASTA batches; SNVs)\n"
        )
    )

    # ------------------------------------------------------------------
    # Common arguments (shared across pipelines)
    # ------------------------------------------------------------------
    parser.add_argument(
        "--variation_mode",
        choices=["before", "after"],
        required=True,
        help="Score sequences before variation (reference) or after variation (mutated).",
    )
    parser.add_argument("--fasta_sequence_path", required=True, type=Path)
    parser.add_argument("--variant_out_file", required=True, type=Path)
    parser.add_argument("--Transformer_path", required=True, type=Path)
    parser.add_argument("--model_save_path", required=True, type=Path)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")

    # ------------------------------------------------------------------
    # Pipeline selection flags
    #
    # Backward compatibility:
    # - If you pass none of these flags, we default to GWAS mode.
    # - `--ribosnitches` is accepted as an alias for `--ribosnitch`.
    # - `--genomic_variants` is accepted as an alias for `--catalog_variants`.
    # ------------------------------------------------------------------
    pipe = parser.add_mutually_exclusive_group(required=False)
    pipe.add_argument(
        "--gwas",
        action="store_true",
        help="Force GWAS window scoring (default if no pipeline flag is provided).",
    )
    pipe.add_argument(
        "--ribosnitch",
        "--ribosnitches",
        dest="ribosnitch",
        action="store_true",
        help="Run ribosnitch scoring (BRIDGE).",
    )
    pipe.add_argument(
        "--catalog_variants",
        "--genomic_variants",
        dest="catalog_variants",
        action="store_true",
        help="Run ClinVar/TCGA/1000G-style FASTA batch scoring (SNVs).",
    )

    # ------------------------------------------------------------------
    # Ribosnitch-specific options
    # ------------------------------------------------------------------
    parser.add_argument(
        "--ribosnitch_after_variation",
        "--ribosnitches_after_variation",
        dest="ribosnitch_after_variation",
        action="store_true",
        help="Force ALT substitution for ribosnitch scoring, regardless of --variation_mode.",
    )
    parser.add_argument(
        "--ribosnitch_out_dir",
        "--ribosnitches_out_dir",
        dest="ribosnitch_out_dir",
        type=Path,
        default=Path("./results/ribosnitches"),
        help="Root output directory for ribosnitch results.",
    )

    # ------------------------------------------------------------------
    # Catalog-variants options (also used by genomic_variants.py)
    # ------------------------------------------------------------------
    parser.add_argument(
        "--model_id_strategy",
        choices=["from_header", "from_fasta_stem"],
        default="from_header",
        help=(
            "How to choose checkpoint name for catalog variants. "
            "from_header: <PROTEIN>_<CELL> parsed from header; "
            "from_fasta_stem: use FASTA filename stem."
        ),
    )
    parser.add_argument(
        "--k",
        type=int,
        default=1,
        help="K-mer / stride parameter forwarded to build_Transformer_embeddings (catalog variants branch).",
    )
    parser.add_argument(
        "--pos_weight",
        type=float,
        default=2.0,
        help="Positive class weight for BCEWithLogitsLoss (catalog variants branch).",
    )
    parser.add_argument(
        "--strict_ref_match",
        action="store_true",
        help="If set, skip records when REF/ALT cannot be matched inside the window (catalog variants branch).",
    )
    parser.add_argument(
        "--disable_off_by_one",
        action="store_true",
        help="Disable +/-1 position fallback when locating the SNV inside the window (catalog variants branch).",
    )

    return parser

def main() -> None:
    """
    Main CLI entry.

    Workflow:
    1) parse args
    2) choose a pipeline based on mutually-exclusive flags
    3) read FASTA -> (headers, sequences)
    4) load model hub (Transformer + BRIDGE checkpoints)
    5) run the selected pipeline and append lines to `--variant_out_file`
    """
    args = build_argparser().parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    device = torch.device(args.device)

    logging.info("Loading FASTA from %s", args.fasta_sequence_path)
    headers, sequences = read_fasta(args.fasta_sequence_path)

    # Pipeline selection validation.
    # - Default: GWAS (for backward compatibility) when no pipeline flag is provided.
    selected = []
    if bool(getattr(args, "gwas", False)):
        selected.append("gwas")
    if bool(getattr(args, "ribosnitch", False)) or bool(getattr(args, "ribosnitch_after_variation", False)):
        selected.append("ribosnitch")
    if bool(getattr(args, "catalog_variants", False)):
        selected.append("catalog_variants")

    if len(selected) > 1:
        raise SystemExit(f"Conflicting pipeline flags: {selected}. Please choose only one.")

    pipeline = selected[0] if selected else "gwas"

    hub = ModelHub(args.Transformer_path, device)

    if pipeline == "catalog_variants":
        logging.info("Running catalog variants pipeline (%s_variation)", args.variation_mode)
        process_sequences_catalog_variants(headers, sequences, args, hub)

    logging.info("Finished. Results appended to %s", args.variant_out_file)

### Catalog variants main loop: score each sequence and append to the output file

The core `process_sequences_catalog_variants(...)` function can be summarized as:

- **Read**: `headers, seqs` (from `read_fasta`)
- **Parse**: use `parse_header_catalog` to extract coordinates and `REF/ALT`
- **Locate**: use `find_variant_index` (optionally with an off-by-one fallback)
- **Choose checkpoint**: default `model_id=<PROTEIN>_<CELL>`, or use `--model_id_strategy from_fasta_stem`
- **Build features**: Transformer embeddings + handcrafted features (e.g. biochemical features)
- **Infer and write**: append one line per record to `--variant_out_file`

#### Key switches that affect input/output behavior

- `--variation_mode before|after`: whether to replace with ALT before scoring
- `--strict_ref_match`: skip the record if the window base at the variant position matches neither REF nor ALT (stricter)
- `--disable_off_by_one`: disable the +/-1 fallback when locating the variant index (stricter and more controlled)
- `--pos_weight`: positive-class weight for the loss (can affect score magnitudes; keep consistent with training/evaluation)

> In practice, if you want to first validate header parsing and variant localization, start **without** `--strict_ref_match` and inspect where warnings concentrate (often a formatting issue).


In [5]:
def process_sequences_catalog_variants(
    headers: List[str],
    seqs: List[str],
    args: argparse.Namespace,
    hub: ModelHub,
) -> None:
    """
    Variant-aware scoring for ClinVar/TCGA/1000G-style FASTA batches (SNVs).

    Inputs
    ------
    headers, seqs:
        Lists returned by `read_fasta(fasta_path)`. `headers[i]` must correspond to `seqs[i]`.

        For this pipeline, each header should include:
          - region token: `chr:start-end(strand)`
          - variant token: `POS:REF>ALT`
        (protein/cell-line tokens are optional but needed for default `model_id` parsing)

    args:
        Parsed CLI arguments. The following fields are used here:
          - variation_mode: "before" or "after"
          - fasta_sequence_path, variant_out_file
          - Transformer_path, model_save_path, device
          - model_id_strategy
          - k, pos_weight, strict_ref_match, disable_off_by_one

    hub:
        `ModelHub` instance used to load BRIDGE checkpoints and hold the torch device.

    Outputs
    -------
    Appends one line per input record to `args.variant_out_file`:
        <header>\tmodel_id=<...>\tmode=<before|after>\tPrediction_score:<float>

    Error handling
    --------------
    - If header parsing fails: logs an error and skips the record.
    - If REF/ALT cannot be matched in the window:
        * logs a warning
        * skips only if `--strict_ref_match` is set
    - If checkpoint cannot be found for the inferred `model_id`: skips the record.
    """
    out_fp = open_output(args.variant_out_file)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(args.pos_weight), device=hub.device))

    with out_fp.open("a") as fout:
        for header_raw, seq in zip(headers, seqs):
            try:
                ph = parse_header_catalog(header_raw)
            except Exception as e:
                logging.error("[catalog_variants] Header parse failed: %s | %s", header_raw, e)
                continue

            ref, alt = ph.ref, ph.alt
            if ph.strand == "-":
                ref, alt = apply_complement(ref), apply_complement(alt)

            idx0, state = find_variant_index(
                seq=seq,
                seq_start=ph.start,
                var_pos=ph.var_pos,
                ref=ref,
                alt=alt,
                try_off_by_one=(not bool(args.disable_off_by_one)),
            )
            if state == "none":
                logging.warning("[catalog_variants] Cannot match REF/ALT at site: %s", ph.header_raw)
                if bool(args.strict_ref_match):
                    continue

            # choose checkpoint name
            if args.model_id_strategy == "from_fasta_stem":
                model_id = Path(args.fasta_sequence_path).stem
            else:
                if not ph.protein or not ph.cell_line:
                    logging.warning("[catalog_variants] Cannot parse protein/cell line for model id: %s", ph.header_raw)
                    continue
                model_id = f"{ph.protein}_{ph.cell_line}"

            model = hub.load_bridge(Path(args.model_save_path), model_id)
            if model is None:
                continue

            # build modified_seq depending on variation_mode and what we see in input
            if args.variation_mode == "before":
                if state == "alt":
                    logging.info("[catalog_variants] Input already ALT at site; scoring as-is in BEFORE: %s", ph.header_raw)
                modified_seq = seq
            else:  # after
                if idx0 is None:
                    modified_seq = seq
                elif state == "ref":
                    modified_seq = substitute_base(seq, idx0, alt)
                else:
                    modified_seq = seq

            emb, _ = build_Transformer_embeddings(
                sequences=[modified_seq],
                transformer_path=str(args.Transformer_path),
                device=hub.device,
                k=int(args.k),
                transpose_to_ch_first=True,
            )
            N = int(emb.shape[0])

            attn = np.zeros((N, 101, 103))
            struct = np.zeros((N, 1, 101))
            motif = np.zeros((N, 1, 101))
            biochem = dealwithdata2(modified_seq).transpose([0, 2, 1])

            dataset = RBPInferDataset(
                embedding=emb,
                attn=attn,
                struct=struct,
                motif=motif,
                biochem=biochem,
            )
            loader = DataLoader(dataset, batch_size=1, shuffle=False)

            score = validate_without_sigmoid(model, hub.device, loader, criterion).item()

            # Output keeps genomic_variants.py style (adds model_id/mode)
            fout.write(
                f"{ph.header_raw}\tmodel_id={model_id}"
                f"\tmode={args.variation_mode}\tPrediction_score:{score:.6f}\n"
            )

## End-to-end run (optional): load the model and write scores

If you already have a working BRIDGE environment (dependencies installed and valid Transformer/checkpoint paths), you can run an end-to-end example as follows:

1. Make sure these three paths exist:
   - `FASTA_PATH`: your FASTA file (ClinVar/TCGA/1000G catalog variants)
   - `TRANSFORMER_PATH`: the Transformer directory (e.g. `RBPformer`)
   - `MODEL_SAVE_PATH`: the BRIDGE checkpoint directory (containing `<model_id>.pth` or equivalent naming)

2. Start with a small file and run `--device cpu` as a smoke test, then switch to `cuda:0` for large-scale scoring.

> The output file is **appended** to. If a file with the same name already exists, new results will be appended at the end.


In [ ]:
%%bash
set -euo pipefail

BRIDGE_HOME=../../../
cd "$BRIDGE_HOME"
 
TRANSFORMER_PATH="./RBPformer"
MODEL_SAVE_PATH="./results/model"
mkdir -p ./results/variant/mut_before_after_score

declare -A FASTA_MAP
FASTA_MAP["1000genomes"]="./dataset_variant/1000genomes_diff.all.fa"

for dataset in "1000genomes"; do
  fasta="${FASTA_MAP[$dataset]}"

  for mode in "before" "after"; do
    out="./results/variant/mut_before_after_score/${dataset}.${mode}.txt"

    python variant_aware.py \
      --catalog_variants \
      --variation_mode "$mode" \
      --fasta_sequence_path "$fasta" \
      --variant_out_file "$out" \
      --Transformer_path "$TRANSFORMER_PATH" \
      --model_save_path "$MODEL_SAVE_PATH"
  done
done

INFO: Loading FASTA from dataset_variant/1000genomes_diff.all.fa
/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
INFO: Running catalog variants pipeline (before_variation)
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
Y